In [1]:
import pandas as pd

In [2]:
# 1. Load dataset
df = pd.read_csv("mobile_phone_sales.csv")

In [3]:
# 2. Check dataset shape
print("Dataset shape:", df.shape)

Dataset shape: (480, 26)


In [4]:
# 3. Show column names
print("\nColumns:")
print(df.columns.tolist())


Columns:
['Sale_ID', 'Date', 'Country', 'City', 'Store_ID', 'Sales_Channel', 'Brand', 'Model', 'Supplier', 'Storage_GB', 'Color', 'Price_USD', 'COGS_USD', 'Units_Sold', 'Discount_pct', 'Revenue_USD', 'Cost_Total_USD', 'Profit_USD', 'Profit_Margin', 'Inventory_Level', 'Promotion', 'Payment_Method', 'Warranty_Type', 'Customer_Rating', 'Returned', 'Salesperson_ID']


In [5]:
# 4. General info
print("\nDataset info:")
print(df.info())


Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 26 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Sale_ID          480 non-null    int64  
 1   Date             480 non-null    object 
 2   Country          480 non-null    object 
 3   City             480 non-null    object 
 4   Store_ID         480 non-null    object 
 5   Sales_Channel    480 non-null    object 
 6   Brand            480 non-null    object 
 7   Model            480 non-null    object 
 8   Supplier         480 non-null    object 
 9   Storage_GB       480 non-null    int64  
 10  Color            480 non-null    object 
 11  Price_USD        480 non-null    float64
 12  COGS_USD         480 non-null    float64
 13  Units_Sold       480 non-null    int64  
 14  Discount_pct     480 non-null    float64
 15  Revenue_USD      480 non-null    float64
 16  Cost_Total_USD   480 non-null    float64
 17  P

In [6]:
# Check missing values per column
missing = df.isnull().sum()

print("Missing values per column:\n")
print(missing.sort_values(ascending=False))

Missing values per column:

Promotion          128
Sale_ID              0
Date                 0
Returned             0
Customer_Rating      0
Warranty_Type        0
Payment_Method       0
Inventory_Level      0
Profit_Margin        0
Profit_USD           0
Cost_Total_USD       0
Revenue_USD          0
Discount_pct         0
Units_Sold           0
COGS_USD             0
Price_USD            0
Color                0
Storage_GB           0
Supplier             0
Model                0
Brand                0
Sales_Channel        0
Store_ID             0
City                 0
Country              0
Salesperson_ID       0
dtype: int64


In [7]:
# 1. Check full row duplicates
duplicates = df.duplicated().sum()
print("Full duplicates:", duplicates)

Full duplicates: 0


In [8]:
# 2. Check duplicate Sale_ID 
duplicate_ids = df["Sale_ID"].duplicated().sum()
print("Duplicate Sale_ID:", duplicate_ids)

Duplicate Sale_ID: 0


In [9]:
# 3. Basic sanity checks

print("\nSanity checks:")

# Negative or zero values (should not exist)
print("Negative or zero Revenue:", (df["Revenue_USD"] <= 0).sum())
print("Negative or zero Units:", (df["Units_Sold"] <= 0).sum())
print("Negative Price:", (df["Price_USD"] <= 0).sum())
print("Negative Profit:", (df["Profit_USD"] < 0).sum())

# Discount range check
print("Invalid Discount (>0.5):", (df["Discount_pct"] > 0.5).sum())

# Rating range check
print("Invalid Rating (<1 or >5):", ((df["Customer_Rating"] < 1) | (df["Customer_Rating"] > 5)).sum())


Sanity checks:
Negative or zero Revenue: 0
Negative or zero Units: 0
Negative Price: 0
Negative Profit: 0
Invalid Discount (>0.5): 0
Invalid Rating (<1 or >5): 0


In [10]:
# 1. Convert Date to datetime
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Check if any dates failed conversion
print("Invalid dates:", df["Date"].isnull().sum())

Invalid dates: 0


In [11]:
# 2. Check date range
print("\nDate range:")
print("Min date:", df["Date"].min())
print("Max date:", df["Date"].max())


Date range:
Min date: 2024-01-01 00:00:00
Max date: 2024-12-31 00:00:00


In [12]:
# 3. Check unique values in key categorical columns

categorical_cols = [
    "Brand", 
    "Sales_Channel", 
    "Country", 
    "Promotion", 
    "Payment_Method"
]

for col in categorical_cols:
    print(f"\nUnique values in {col}:")
    print(df[col].value_counts(dropna=False))


Unique values in Brand:
Brand
Apple      107
Xiaomi     104
Google     104
OnePlus     85
Samsung     80
Name: count, dtype: int64

Unique values in Sales_Channel:
Sales_Channel
Store     258
Online    222
Name: count, dtype: int64

Unique values in Country:
Country
Ukraine    109
USA        104
Poland      98
UK          90
Germany     79
Name: count, dtype: int64

Unique values in Promotion:
Promotion
NaN               128
Black Friday      120
Holiday Sale      117
Back to School    115
Name: count, dtype: int64

Unique values in Payment_Method:
Payment_Method
PayPal         108
Cash           106
Credit Card     93
Apple Pay       88
Google Pay      85
Name: count, dtype: int64


In [13]:
# Replace NaN in Promotion with "No Promotion"
df["Promotion"] = df["Promotion"].fillna("No Promotion")

# Check again
print(df["Promotion"].value_counts())

Promotion
No Promotion      128
Black Friday      120
Holiday Sale      117
Back to School    115
Name: count, dtype: int64


In [14]:
# Extract date features
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Month_Name"] = df["Date"].dt.strftime("%B")

# Check result
print(df[["Date", "Year", "Month", "Month_Name"]].head())

        Date  Year  Month Month_Name
0 2024-05-03  2024      5        May
1 2024-10-13  2024     10    October
2 2024-04-02  2024      4      April
3 2024-10-12  2024     10    October
4 2024-01-12  2024      1    January


In [15]:
# 1. Discount flag
df["Is_Discounted"] = df["Discount_pct"] > 0

# 2. Return flag (перетворимо в більш читабельний вигляд)
df["Is_Returned"] = df["Returned"] == 1

# 3. High margin flag
df["Is_High_Margin"] = df["Profit_Margin"] > 0.3

# 4. Revenue per unit
df["Revenue_per_Unit"] = df["Revenue_USD"] / df["Units_Sold"]

# Check result
print(df[[
    "Discount_pct", "Is_Discounted",
    "Returned", "Is_Returned",
    "Profit_Margin", "Is_High_Margin",
    "Revenue_per_Unit"
]].head())

   Discount_pct  Is_Discounted  Returned  Is_Returned  Profit_Margin  \
0          0.00          False         0        False          0.333   
1          0.10           True         0        False          0.198   
2          0.15           True         0        False          0.170   
3          0.10           True         0        False          0.221   
4          0.05           True         0        False          0.356   

   Is_High_Margin  Revenue_per_Unit  
0            True        384.720000  
1           False        307.726667  
2           False        661.767500  
3           False        608.680000  
4            True        910.137500  


In [16]:
# Headline business KPIs
headline_kpis = {
    "total_revenue": round(df["Revenue_USD"].sum(), 2),
    "total_profit": round(df["Profit_USD"].sum(), 2),
    "overall_profit_margin": round(df["Profit_USD"].sum() / df["Revenue_USD"].sum(), 4),
    "total_units_sold": int(df["Units_Sold"].sum()),
    "total_transactions": int(df["Sale_ID"].nunique()),
    "average_order_value": round(df["Revenue_USD"].mean(), 2),
    "average_units_per_transaction": round(df["Units_Sold"].mean(), 2),
    "average_selling_price_per_unit": round(df["Revenue_USD"].sum() / df["Units_Sold"].sum(), 2),
    "return_rate": round(df["Is_Returned"].mean(), 4),
    "discounted_sales_share": round(df["Is_Discounted"].mean(), 4)
}

print(headline_kpis)

{'total_revenue': np.float64(830708.7), 'total_profit': np.float64(258637.91), 'overall_profit_margin': np.float64(0.3113), 'total_units_sold': 1177, 'total_transactions': 480, 'average_order_value': np.float64(1730.64), 'average_units_per_transaction': np.float64(2.45), 'average_selling_price_per_unit': np.float64(705.78), 'return_rate': np.float64(0.225), 'discounted_sales_share': np.float64(0.5833)}


In [17]:
# Brand-level aggregation
brand_df = df.groupby("Brand").agg({
    "Revenue_USD": "sum",
    "Profit_USD": "sum",
    "Units_Sold": "sum",
    "Sale_ID": "nunique",
    "Is_Returned": "mean",
    "Customer_Rating": "mean"
}).reset_index()

# Add profit margin
brand_df["Profit_Margin"] = brand_df["Profit_USD"] / brand_df["Revenue_USD"]

# Rename columns for clarity
brand_df = brand_df.rename(columns={
    "Sale_ID": "Transactions",
    "Is_Returned": "Return_Rate",
    "Customer_Rating": "Avg_Rating"
})

# Round values
brand_df = brand_df.round({
    "Revenue_USD": 2,
    "Profit_USD": 2,
    "Profit_Margin": 4,
    "Return_Rate": 4,
    "Avg_Rating": 2
})

# Convert to JSON-ready format
brand_performance = brand_df.sort_values(by="Revenue_USD", ascending=False).to_dict(orient="records")

# Preview
print(brand_performance[:3])

[{'Brand': 'Apple', 'Revenue_USD': 202000.21, 'Profit_USD': 63793.35, 'Units_Sold': 278, 'Transactions': 107, 'Return_Rate': 0.2056, 'Avg_Rating': 4.32, 'Profit_Margin': 0.3158}, {'Brand': 'Xiaomi', 'Revenue_USD': 173442.96, 'Profit_USD': 55421.13, 'Units_Sold': 233, 'Transactions': 104, 'Return_Rate': 0.25, 'Avg_Rating': 4.25, 'Profit_Margin': 0.3195}, {'Brand': 'Google', 'Revenue_USD': 172462.84, 'Profit_USD': 52836.88, 'Units_Sold': 251, 'Transactions': 104, 'Return_Rate': 0.2212, 'Avg_Rating': 4.39, 'Profit_Margin': 0.3064}]


In [18]:
# Channel-level aggregation
channel_df = df.groupby("Sales_Channel").agg({
    "Revenue_USD": "sum",
    "Profit_USD": "sum",
    "Units_Sold": "sum",
    "Sale_ID": "nunique",
    "Is_Returned": "mean",
    "Customer_Rating": "mean",
    "Revenue_per_Unit": "mean"
}).reset_index()

# Add profit margin
channel_df["Profit_Margin"] = channel_df["Profit_USD"] / channel_df["Revenue_USD"]

# Rename columns
channel_df = channel_df.rename(columns={
    "Sale_ID": "Transactions",
    "Is_Returned": "Return_Rate",
    "Customer_Rating": "Avg_Rating"
})

# Round values
channel_df = channel_df.round({
    "Revenue_USD": 2,
    "Profit_USD": 2,
    "Profit_Margin": 4,
    "Return_Rate": 4,
    "Avg_Rating": 2,
    "Revenue_per_Unit": 2
})

# Convert to JSON-ready format
channel_performance = channel_df.sort_values(by="Revenue_USD", ascending=False).to_dict(orient="records")

# Preview
print(channel_performance)

[{'Sales_Channel': 'Store', 'Revenue_USD': 449884.81, 'Profit_USD': 136804.67, 'Units_Sold': 642, 'Transactions': 258, 'Return_Rate': 0.2403, 'Avg_Rating': 4.32, 'Revenue_per_Unit': 704.32, 'Profit_Margin': 0.3041}, {'Sales_Channel': 'Online', 'Revenue_USD': 380823.89, 'Profit_USD': 121833.24, 'Units_Sold': 535, 'Transactions': 222, 'Return_Rate': 0.2072, 'Avg_Rating': 4.3, 'Revenue_per_Unit': 715.49, 'Profit_Margin': 0.3199}]


In [19]:
# Promotion-level aggregation
promo_df = df.groupby("Promotion").agg({
    "Revenue_USD": "sum",
    "Profit_USD": "sum",
    "Units_Sold": "sum",
    "Sale_ID": "nunique",
    "Is_Returned": "mean",
    "Discount_pct": "mean"
}).reset_index()

# Add profit margin
promo_df["Profit_Margin"] = promo_df["Profit_USD"] / promo_df["Revenue_USD"]

# Rename columns
promo_df = promo_df.rename(columns={
    "Sale_ID": "Transactions",
    "Is_Returned": "Return_Rate",
    "Discount_pct": "Avg_Discount"
})

# Round values
promo_df = promo_df.round({
    "Revenue_USD": 2,
    "Profit_USD": 2,
    "Profit_Margin": 4,
    "Return_Rate": 4,
    "Avg_Discount": 3
})

# Sort by Revenue
promo_df = promo_df.sort_values(by="Revenue_USD", ascending=False)

# Convert to JSON-ready format
promotion_performance = promo_df.to_dict(orient="records")

# Preview
print(promotion_performance)

[{'Promotion': 'No Promotion', 'Revenue_USD': 230856.85, 'Profit_USD': 74087.61, 'Units_Sold': 327, 'Transactions': 128, 'Return_Rate': 0.2656, 'Avg_Discount': 0.056, 'Profit_Margin': 0.3209}, {'Promotion': 'Holiday Sale', 'Revenue_USD': 204826.89, 'Profit_USD': 65289.67, 'Units_Sold': 294, 'Transactions': 117, 'Return_Rate': 0.2137, 'Avg_Discount': 0.056, 'Profit_Margin': 0.3188}, {'Promotion': 'Black Friday', 'Revenue_USD': 202127.09, 'Profit_USD': 58854.37, 'Units_Sold': 282, 'Transactions': 120, 'Return_Rate': 0.2083, 'Avg_Discount': 0.063, 'Profit_Margin': 0.2912}, {'Promotion': 'Back to School', 'Revenue_USD': 192897.87, 'Profit_USD': 60406.26, 'Units_Sold': 274, 'Transactions': 115, 'Return_Rate': 0.2087, 'Avg_Discount': 0.057, 'Profit_Margin': 0.3132}]


In [20]:
# Model-level aggregation
model_df = df.groupby(["Brand", "Model"]).agg({
    "Revenue_USD": "sum",
    "Profit_USD": "sum",
    "Units_Sold": "sum",
    "Sale_ID": "nunique",
    "Is_Returned": "mean",
    "Customer_Rating": "mean",
    "Revenue_per_Unit": "mean"
}).reset_index()

# Add profit margin
model_df["Profit_Margin"] = model_df["Profit_USD"] / model_df["Revenue_USD"]

# Rename columns
model_df = model_df.rename(columns={
    "Sale_ID": "Transactions",
    "Is_Returned": "Return_Rate",
    "Customer_Rating": "Avg_Rating"
})

# Round values
model_df = model_df.round({
    "Revenue_USD": 2,
    "Profit_USD": 2,
    "Profit_Margin": 4,
    "Return_Rate": 4,
    "Avg_Rating": 2,
    "Revenue_per_Unit": 2
})

# Sort by revenue
model_df = model_df.sort_values(by="Revenue_USD", ascending=False)

# Convert to JSON-ready format
model_performance = model_df.to_dict(orient="records")

# Preview first rows
print(model_performance[:5])

[{'Brand': 'Google', 'Model': 'Pixel 8', 'Revenue_USD': 92015.62, 'Profit_USD': 27613.18, 'Units_Sold': 139, 'Transactions': 59, 'Return_Rate': 0.1695, 'Avg_Rating': 4.35, 'Revenue_per_Unit': 676.31, 'Profit_Margin': 0.3001}, {'Brand': 'OnePlus', 'Model': 'OnePlus 11', 'Revenue_USD': 83534.06, 'Profit_USD': 24816.48, 'Units_Sold': 126, 'Transactions': 43, 'Return_Rate': 0.186, 'Avg_Rating': 4.35, 'Revenue_per_Unit': 671.87, 'Profit_Margin': 0.2971}, {'Brand': 'Google', 'Model': 'Pixel 7', 'Revenue_USD': 80447.22, 'Profit_USD': 25223.7, 'Units_Sold': 112, 'Transactions': 45, 'Return_Rate': 0.2889, 'Avg_Rating': 4.43, 'Revenue_per_Unit': 752.24, 'Profit_Margin': 0.3135}, {'Brand': 'Apple', 'Model': 'iPhone 13', 'Revenue_USD': 73446.19, 'Profit_USD': 24982.16, 'Units_Sold': 97, 'Transactions': 36, 'Return_Rate': 0.2222, 'Avg_Rating': 4.39, 'Revenue_per_Unit': 747.15, 'Profit_Margin': 0.3401}, {'Brand': 'Xiaomi', 'Model': 'Redmi Note 12', 'Revenue_USD': 70260.69, 'Profit_USD': 22051.11, 'U

In [21]:
# Monthly aggregation
monthly_df = df.groupby(["Year", "Month", "Month_Name"]).agg({
    "Revenue_USD": "sum",
    "Profit_USD": "sum",
    "Units_Sold": "sum",
    "Sale_ID": "nunique",
    "Is_Returned": "mean"
}).reset_index()

# Add profit margin
monthly_df["Profit_Margin"] = monthly_df["Profit_USD"] / monthly_df["Revenue_USD"]

# Rename columns
monthly_df = monthly_df.rename(columns={
    "Sale_ID": "Transactions",
    "Is_Returned": "Return_Rate"
})

# Sort correctly by Year + Month
monthly_df = monthly_df.sort_values(by=["Year", "Month"])

# Round values
monthly_df = monthly_df.round({
    "Revenue_USD": 2,
    "Profit_USD": 2,
    "Profit_Margin": 4,
    "Return_Rate": 4
})

# Convert to JSON-ready format
monthly_performance = monthly_df.to_dict(orient="records")

# Preview
print(monthly_performance[:5])

[{'Year': 2024, 'Month': 1, 'Month_Name': 'January', 'Revenue_USD': 85111.27, 'Profit_USD': 27020.25, 'Units_Sold': 118, 'Transactions': 46, 'Return_Rate': 0.2391, 'Profit_Margin': 0.3175}, {'Year': 2024, 'Month': 2, 'Month_Name': 'February', 'Revenue_USD': 68870.79, 'Profit_USD': 20099.24, 'Units_Sold': 99, 'Transactions': 37, 'Return_Rate': 0.0811, 'Profit_Margin': 0.2918}, {'Year': 2024, 'Month': 3, 'Month_Name': 'March', 'Revenue_USD': 82608.42, 'Profit_USD': 24179.54, 'Units_Sold': 117, 'Transactions': 50, 'Return_Rate': 0.26, 'Profit_Margin': 0.2927}, {'Year': 2024, 'Month': 4, 'Month_Name': 'April', 'Revenue_USD': 62589.24, 'Profit_USD': 21139.18, 'Units_Sold': 92, 'Transactions': 37, 'Return_Rate': 0.1892, 'Profit_Margin': 0.3377}, {'Year': 2024, 'Month': 5, 'Month_Name': 'May', 'Revenue_USD': 60557.04, 'Profit_USD': 18921.66, 'Units_Sold': 89, 'Transactions': 38, 'Return_Rate': 0.1053, 'Profit_Margin': 0.3125}]


In [22]:
# Country-level aggregation
country_df = df.groupby("Country").agg({
    "Revenue_USD": "sum",
    "Profit_USD": "sum",
    "Units_Sold": "sum",
    "Sale_ID": "nunique",
    "Is_Returned": "mean"
}).reset_index()

# Add profit margin
country_df["Profit_Margin"] = country_df["Profit_USD"] / country_df["Revenue_USD"]

# Rename columns
country_df = country_df.rename(columns={
    "Sale_ID": "Transactions",
    "Is_Returned": "Return_Rate"
})

# Round values
country_df = country_df.round({
    "Revenue_USD": 2,
    "Profit_USD": 2,
    "Profit_Margin": 4,
    "Return_Rate": 4
})

# Sort by revenue
country_df = country_df.sort_values(by="Revenue_USD", ascending=False)

# Convert to JSON-ready format
country_performance = country_df.to_dict(orient="records")

# --- City-level (optional but powerful) ---

city_df = df.groupby(["Country", "City"]).agg({
    "Revenue_USD": "sum",
    "Profit_USD": "sum",
    "Units_Sold": "sum",
    "Sale_ID": "nunique"
}).reset_index()

city_df["Profit_Margin"] = city_df["Profit_USD"] / city_df["Revenue_USD"]

city_df = city_df.rename(columns={
    "Sale_ID": "Transactions"
})

city_df = city_df.round({
    "Revenue_USD": 2,
    "Profit_USD": 2,
    "Profit_Margin": 4
})

city_df = city_df.sort_values(by="Revenue_USD", ascending=False)

city_performance = city_df.to_dict(orient="records")

# Preview
print(country_performance)
print(city_performance[:5])

[{'Country': 'Ukraine', 'Revenue_USD': 200771.46, 'Profit_USD': 63069.97, 'Units_Sold': 281, 'Transactions': 109, 'Return_Rate': 0.1835, 'Profit_Margin': 0.3141}, {'Country': 'USA', 'Revenue_USD': 176659.16, 'Profit_USD': 54940.56, 'Units_Sold': 253, 'Transactions': 104, 'Return_Rate': 0.2212, 'Profit_Margin': 0.311}, {'Country': 'Poland', 'Revenue_USD': 172787.94, 'Profit_USD': 53203.8, 'Units_Sold': 249, 'Transactions': 98, 'Return_Rate': 0.2959, 'Profit_Margin': 0.3079}, {'Country': 'UK', 'Revenue_USD': 147147.73, 'Profit_USD': 46206.76, 'Units_Sold': 205, 'Transactions': 90, 'Return_Rate': 0.2444, 'Profit_Margin': 0.314}, {'Country': 'Germany', 'Revenue_USD': 133342.41, 'Profit_USD': 41216.82, 'Units_Sold': 189, 'Transactions': 79, 'Return_Rate': 0.1772, 'Profit_Margin': 0.3091}]
[{'Country': 'Ukraine', 'City': 'Lviv', 'Revenue_USD': 114066.88, 'Profit_USD': 35860.47, 'Units_Sold': 156, 'Transactions': 61, 'Profit_Margin': 0.3144}, {'Country': 'Poland', 'City': 'Warsaw', 'Revenue_U

In [23]:
import json

report_data = {
    "headline_kpis": headline_kpis,
    "brand_performance": brand_performance,
    "channel_performance": channel_performance,
    "promotion_performance": promotion_performance,
    "model_performance": model_performance,
    "monthly_performance": monthly_performance,
    "country_performance": country_performance,
    "city_performance": city_performance
}

# Preview keys
print(report_data.keys())

# Pretty preview of first part
print(json.dumps(report_data["headline_kpis"], indent=4))

# Save to JSON file
with open("mobile_store_report_data.json", "w", encoding="utf-8") as f:
    json.dump(report_data, f, indent=4, ensure_ascii=False)

print("\nJSON file saved successfully.")

dict_keys(['headline_kpis', 'brand_performance', 'channel_performance', 'promotion_performance', 'model_performance', 'monthly_performance', 'country_performance', 'city_performance'])
{
    "total_revenue": 830708.7,
    "total_profit": 258637.91,
    "overall_profit_margin": 0.3113,
    "total_units_sold": 1177,
    "total_transactions": 480,
    "average_order_value": 1730.64,
    "average_units_per_transaction": 2.45,
    "average_selling_price_per_unit": 705.78,
    "return_rate": 0.225,
    "discounted_sales_share": 0.5833
}

JSON file saved successfully.


In [24]:
# Top / bottom brand summaries
top_brand_by_revenue = max(brand_performance, key=lambda x: x["Revenue_USD"])
top_brand_by_profit = max(brand_performance, key=lambda x: x["Profit_USD"])
worst_brand_by_return_rate = max(brand_performance, key=lambda x: x["Return_Rate"])

# Top / bottom model summaries
top_5_models_by_revenue = sorted(model_performance, key=lambda x: x["Revenue_USD"], reverse=True)[:5]
top_5_models_by_profit = sorted(model_performance, key=lambda x: x["Profit_USD"], reverse=True)[:5]
worst_5_models_by_margin = sorted(model_performance, key=lambda x: x["Profit_Margin"])[:5]

# Best / worst promotion by profit
best_promotion_by_profit = max(promotion_performance, key=lambda x: x["Profit_USD"])
worst_promotion_by_margin = min(promotion_performance, key=lambda x: x["Profit_Margin"])

# Best / worst month
best_month_by_revenue = max(monthly_performance, key=lambda x: x["Revenue_USD"])
worst_month_by_revenue = min(monthly_performance, key=lambda x: x["Revenue_USD"])

# Best country
top_country_by_revenue = max(country_performance, key=lambda x: x["Revenue_USD"])
top_country_by_profit = max(country_performance, key=lambda x: x["Profit_USD"])

# Compact package for AI
ai_report_data = {
    "headline_kpis": headline_kpis,
    "top_brand_by_revenue": top_brand_by_revenue,
    "top_brand_by_profit": top_brand_by_profit,
    "worst_brand_by_return_rate": worst_brand_by_return_rate,
    "channel_performance": channel_performance,
    "best_promotion_by_profit": best_promotion_by_profit,
    "worst_promotion_by_margin": worst_promotion_by_margin,
    "top_5_models_by_revenue": top_5_models_by_revenue,
    "top_5_models_by_profit": top_5_models_by_profit,
    "worst_5_models_by_margin": worst_5_models_by_margin,
    "best_month_by_revenue": best_month_by_revenue,
    "worst_month_by_revenue": worst_month_by_revenue,
    "top_country_by_revenue": top_country_by_revenue,
    "top_country_by_profit": top_country_by_profit
}

# Preview
import json
print(json.dumps(ai_report_data, indent=4))

{
    "headline_kpis": {
        "total_revenue": 830708.7,
        "total_profit": 258637.91,
        "overall_profit_margin": 0.3113,
        "total_units_sold": 1177,
        "total_transactions": 480,
        "average_order_value": 1730.64,
        "average_units_per_transaction": 2.45,
        "average_selling_price_per_unit": 705.78,
        "return_rate": 0.225,
        "discounted_sales_share": 0.5833
    },
    "top_brand_by_revenue": {
        "Brand": "Apple",
        "Revenue_USD": 202000.21,
        "Profit_USD": 63793.35,
        "Units_Sold": 278,
        "Transactions": 107,
        "Return_Rate": 0.2056,
        "Avg_Rating": 4.32,
        "Profit_Margin": 0.3158
    },
    "top_brand_by_profit": {
        "Brand": "Apple",
        "Revenue_USD": 202000.21,
        "Profit_USD": 63793.35,
        "Units_Sold": 278,
        "Transactions": 107,
        "Return_Rate": 0.2056,
        "Avg_Rating": 4.32,
        "Profit_Margin": 0.3158
    },
    "worst_brand_by_return_ra

In [25]:
import json

# Save compact AI-ready dataset
with open("mobile_store_ai_report_data.json", "w", encoding="utf-8") as f:
    json.dump(ai_report_data, f, indent=4, ensure_ascii=False)

print("AI-ready JSON saved successfully.")

AI-ready JSON saved successfully.


In [26]:
import json

# Load compact AI-ready JSON
with open("mobile_store_ai_report_data.json", "r", encoding="utf-8") as f:
    ai_report_data = json.load(f)

# Convert JSON to formatted string for prompt
json_payload = json.dumps(ai_report_data, indent=2)

# System prompt
system_prompt = """
You are a senior business analyst preparing an executive business performance report for the owner of a mobile phone retail business.

Your task is to analyze the provided KPI data and write a clear, structured, decision-oriented report for management.

Rules:
1. Use ONLY the data provided.
2. Do NOT invent facts, numbers, explanations, or trends not supported by the data.
3. Do NOT mention JSON, Python, scripts, code, or data processing.
4. Focus on business meaning, not technical explanation.
5. Write in a professional executive tone.
6. Be concise, specific, and practical.
7. Highlight both strengths and risks.
8. Recommendations must be actionable and directly connected to the data.
9. If something is not fully supported by the data, state it carefully.
10. Return the report in Markdown format only.
"""

# User prompt
user_prompt = f"""
Prepare an executive business performance report for a mobile phone retail business based on the KPI summary below.

Required structure:

# Executive Summary
Provide a short management-level summary of overall business performance.

# Key Business Insights
Identify the most important findings across:
- overall performance
- brands
- sales channels
- promotions
- products/models
- monthly trend
- geography

# Risks and Weaknesses
Highlight the most important business risks, weak areas, or underperforming segments.

# Recommendations
Provide 4-6 specific business recommendations based only on the data.

Important:
- Focus on profit, margin, return rate, and business performance quality — not only revenue.
- Do not repeat raw numbers excessively.
- Interpret what the numbers mean for decision-making.
- Keep the report practical and management-oriented.

KPI DATA:
{json_payload}
"""

print("SYSTEM PROMPT:\n")
print(system_prompt)

print("\n" + "="*80 + "\n")

print("USER PROMPT:\n")
print(user_prompt)

SYSTEM PROMPT:


You are a senior business analyst preparing an executive business performance report for the owner of a mobile phone retail business.

Your task is to analyze the provided KPI data and write a clear, structured, decision-oriented report for management.

Rules:
1. Use ONLY the data provided.
2. Do NOT invent facts, numbers, explanations, or trends not supported by the data.
3. Do NOT mention JSON, Python, scripts, code, or data processing.
4. Focus on business meaning, not technical explanation.
5. Write in a professional executive tone.
6. Be concise, specific, and practical.
7. Highlight both strengths and risks.
8. Recommendations must be actionable and directly connected to the data.
9. If something is not fully supported by the data, state it carefully.
10. Return the report in Markdown format only.



USER PROMPT:


Prepare an executive business performance report for a mobile phone retail business based on the KPI summary below.

Required structure:

# Executive 

In [34]:
import json
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load env variables
load_dotenv()

# Initialize client ONCE
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Generate report
response = client.responses.create(
    model="gpt-5.2",
    input=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.3
)

# Extract result
report_markdown = response.output_text

print(report_markdown)

# Executive Summary
The business delivered **strong profitability** on solid sales volume, with an **overall profit margin of ~31%** and **$258.6k profit** on **$830.7k revenue**. Customer satisfaction appears healthy (ratings ~4.3 across key cuts), but **returns are high (22.5%)** and **discounting is heavily used (58.3% of sales)**—both of which can erode profit quality if not actively managed. Performance is broadly balanced across Store and Online, with **Online showing better margin and lower returns**, while **Store drives more revenue but with higher returns**.

# Key Business Insights

## Overall performance
- Profitability is a clear strength: **~31% overall margin** indicates good gross profit capture relative to revenue.
- Basket economics are favorable: **AOV ~$1.73k** and **~2.45 units per transaction** suggest effective multi-unit selling.
- **Return rate (22.5%)** is a material drag on performance quality and operational cost exposure.

## Brands
- **Apple is the leading

In [36]:
# Save Markdown report
file_path = "business_report.md"

with open(file_path, "w", encoding="utf-8") as f:
    f.write(report_markdown)

print(f"Report saved to {file_path}")

Report saved to business_report.md


In [37]:
# Create Quarto file from AI report
with open("report.qmd", "w", encoding="utf-8") as f:
    f.write(f"""---
title: "Mobile Store Business Report"
format: pdf
toc: true
number-sections: true
---

{report_markdown}
""")

print("report.qmd created")

report.qmd created
